# Gasificador — Catálogo de condiciones de contorno y configuraciones

Módulos probados: `src.units.gasifier.config.*`  
Runner: `src.solvers.runner_gasifier`  
Combustible: `softwood_spruce.yaml`

Este notebook NO ejecuta ninguna integración temporal. Su propósito es verificar
que todos los bloques `build_*` del gasificador funcionan correctamente y mostrar
el contenido de cada dict de configuración antes de usarlo en una simulación real.

## Ejes de configuración del gasificador

| Eje | Módulo | Opciones disponibles |
|---|---|---|
| **Condiciones de contorno** (gas + sólido) | `gasifier.config.boundary_c` | `v_gas_in`, `v_out`, `v_solid`, `direction`, `inlet_mode` |
| **Condición térmica de pared** | `gasifier.config.thermal_bc` | `adiabatic`, `heatfluxwall`, `fixed_twall`, `ambient_htc` |
| **Coeficientes de transporte** | `gasifier.config.transport` | `constant`, `correlation` |
| **Propiedades del gas** | `gasifier.config.gas_props` | `fixed`, `constant`, `polynomial` |
| **Modelo de pared dinámica** | `gasifier.config.wall_c` | sin `wall_config` (16·N) / con `wall_config` (17·N) |

## Puntos de operación — determinados implícitamente por las BC

| Punto de operación | `v_gas_in` | `v_out` | `v_solid` | `direction` | Descripción física |
|---|---|---|---|---|---|
| **batch** | None | `0.0` | 0 | — | Sistema sellado, sin flujo de gas |
| **semibatch** | None | `>0` | 0 | — | Sin entrada; venteo si P > P_out (v_out = v_vent_max) |
| **CSTR / paso gas** | > 0 | `None` | 0 | — | v_out calculado por continuidad molar |
| **updraft** | > 0 | `None` | > 0 | `"updraft"` | Contracorriente gas↑ / sólido↓ |
| **downdraft** | > 0 | `None` | > 0 | `"downdraft"` | Cocorriente gas↓ / sólido↓ |
| **conveyor** | > 0 | `None` | > 0 | `"updraft"`/`"downdraft"` | Sólido calculado del balance de masa |

## Condiciones térmicas de pared — descripción física

| Modo | Qué prescribe | Parámetros requeridos |
|---|---|---|
| `adiabatic` | Sin intercambio lateral | — |
| `heatfluxwall` | Potencia total [W] uniforme o por celda | `Qwall` |
| `fixed_twall` | Temperatura de pared constante [K] | `T_wall` |
| `ambient_htc` | Convección externa + conducción de pared | `h_ambi`, `T_ambi`, `k_wall` |
| `wall_config` (shell-tube) | Temperatura de pared como variable de estado Tw(z,t) | `material`, `Di`, `Do`, `T_w_init` |

---

## Tests realizados

1. **Condiciones de contorno** — `build_bc_config` para los 6 puntos de operación.
2. **Condiciones térmicas de pared** — `build_thermal_bc_config` para los 4 modos.
3. **Transporte** — `build_transport_config` en modos `constant` y `correlation`.
4. **Propiedades del gas** — `build_gas_prop_config` en los 3 modos.
5. **Construcción y validación del dict `params` completo** — casos representativos; verificación con `_validate_gasifier_params` sin integrar.

In [7]:
import os, sys
import numpy as np
import pandas as pd

ROOT = os.path.abspath(os.path.join(os.getcwd(), "../.."))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from src.io.fuels_reader                     import read_fueldb
from src.units.gasifier.config.boundary_c    import build_bc_config
from src.units.gasifier.config.thermal_bc    import build_thermal_bc_config
from src.units.gasifier.config.transport     import build_transport_config
from src.units.gasifier.config.gas_props     import build_gas_prop_config, GASIFIER_GAS_SPECIES
from src.units.gasifier.config.solid_props   import build_solid_prop_config
from src.units.gasifier.config.initial_c     import build_initial_c_config
from src.solvers.runner_gasifier             import _validate_gasifier_params

# Rutas a bases de datos (relativas al ROOT del proyecto)
FUEL_PATH = os.path.join(ROOT, "materials", "fuels", "softwood_spruce.yaml")
GAS_DB    = os.path.join(ROOT, "materials", "fluids", "gasdb.txt")
SOLID_DB  = os.path.join(ROOT, "materials", "solids", "soliddb.txt")

# Combustible y geometría de referencia (usados en todos los tests)
fuel_config = read_fueldb(FUEL_PATH)
nc          = 9
species     = list(GASIFIER_GAS_SPECIES)   # ['CO','CO2','H2O','H2','O2','CH4','C2H4','tar','N2']

Di, Do  = 0.10, 0.114    # [m] diámetro interno / externo (e_pared = 7 mm, acero inox)
e_wall  = (Do - Di) / 2  # [m] espesor de pared
L       = 0.50           # [m] longitud del reactor
N       = 5              # [-] número de celdas (1D con N=5 nodos)
dz      = L / N          # [m] tamaño de celda
Ai      = 0.25 * np.pi * Di**2   # [m²] sección transversal interna
Pi, Po  = np.pi * Di, np.pi * Do  # [m]  perímetros interno y externo
epsi_r  = 0.60           # [-]  porosidad del lecho (fracción de huecos)
dp0     = float(fuel_config["physical"]["dp_initial"])          # [m]  diámetro inicial de partícula
rho_p   = float(fuel_config["physical"]["rho_particle"])        # [kg/m³] densidad de partícula
rho_char0 = fuel_config["pyrolysis_yields"]["char"] * rho_p * (1 - epsi_r)  # [kg/m³_bed] densidad de ref. del char

print("Imports OK")
print(f"Combustible : {fuel_config['fuel_id']}")
print(f"Especies    : {species}")
print(f"Geometría   : Di={Di} m  Do={Do} m  L={L} m  N={N}  dz={dz:.3f} m")

Imports OK
Combustible : softwood_spruce
Especies    : ['CO', 'CO2', 'H2O', 'H2', 'O2', 'CH4', 'C2H4', 'tar', 'N2']
Geometría   : Di=0.1 m  Do=0.114 m  L=0.5 m  N=5  dz=0.100 m


## TEST 1 — Condiciones de contorno (`build_bc_config`)

El modo operativo del gasificador **no se especifica explícitamente**. Emerge de la
combinación de `v_gas_in`, `v_out`, `Cv` y `v_solid`.

| Punto de operación | `v_gas_in` | `v_out` | `Cv` | `v_solid` | `direction` | `inlet_mode` |
|---|---|---|---|---|---|---|
| **batch** (sellado) | `None` | `0.0` | None | 0 | — | — |
| **semibatch simple** | `None` | `> 0` | None | 0 | — | — |
| **semibatch ISA** | `None` | None | `> 0` | 0 | — | — |
| **CSTR / paso gas** | `> 0` | `None` | None | 0 | — | — |
| **CSTR + válvula ISA** | `> 0` | None | `> 0` | 0 | — | — |
| **updraft** | `> 0` | `None` | None | `> 0` | `"updraft"` | `"prescribed"` |
| **downdraft** | `> 0` | `None` | None | `> 0` | `"downdraft"` | `"prescribed"` |
| **conveyor** | `> 0` | `None` | None | `> 0` | `"updraft/downdraft"` | `"computed"` |

### Semántica del outlet (`v_out` y `Cv` son mutuamente excluyentes para venteo)

| Parámetro | Valor | Comportamiento | Caso típico |
|-----------|-------|----------------|-------------|
| `v_out` | `0.0` | `v_out = 0` siempre — sistema sellado | Batch pirólisis |
| `v_out` | `> 0` | `v_out = max(0,(P-P_out)/P_out)·v_out` — venteo lineal | Semibatch simple |
| `v_out` | `None` | `v_out = v_in·Ctot_in/Ctot_out` — continuidad molar | CSTR, updraft, conveyor |
| `Cv` | `> 0` | ISA-75.01: `Q ∝ Cv·√(ΔP·P_up/(T·Sg))` — densidad-dependiente | Semibatch/CSTR con válvula real |

**Diferencia clave:** `v_out` es una velocidad máxima [m/s] intuitiva para exploración paramétrica;
`Cv` es el coeficiente de caudal ISA [-] que corresponde a especificaciones industriales reales.

### Condiciones de entrada usadas en los tests

| Parámetro | Valor | Descripción |
|---|---|---|
| `P_OUT` | 1.01325 bar | Presión de salida de referencia |
| `V_IN` | 0.05 m/s | Velocidad superficial del gas de entrada |
| `T_IN` | 500 K | Temperatura del gas de entrada |
| `y_air` | 21 % O₂ + 79 % N₂ | Composición de referencia (aire seco) |
| `V_SOL` | 1.5×10⁻⁴ m/s | Velocidad superficial del sólido |
| `T_SOL` | 300 K | Temperatura del sólido en la entrada |

In [ ]:
P_OUT = 1.01325    # [bar]   presión de salida del reactor (condición de contorno aguas abajo)
V_IN  = 0.05       # [m/s]   velocidad superficial del gas de entrada al reactor
T_IN  = 500.0      # [K]     temperatura del gas de entrada (precalentado a 227 °C)

# Composición del gas de entrada: aire como agente gasificante de referencia
y_air = np.zeros(nc)
y_air[species.index("O2")] = 0.21
y_air[species.index("N2")] = 0.79

V_SOL = 1.5e-4     # [m/s]   velocidad superficial del sólido
T_SOL = 300.0      # [K]     temperatura del sólido en la entrada
rho_sol_in = np.array([rho_p*(1-epsi_r), 0.0, 0.0])   # [kg/m³_bed] [biomasa, char, humedad]

# ── batch: sin flujo de gas, sistema sellado (v_out=0.0) ─────────────────────
bc_batch = build_bc_config(n_comp=nc, P_out_bar=P_OUT, v_out=0.0)

# ── semibatch simple: venteo proporcional al exceso de presión ─────────────────
bc_semibatch = build_bc_config(n_comp=nc, P_out_bar=P_OUT, v_out=0.10)

# ── semibatch ISA: válvula ISA-75.01 (densidad y temperatura dependientes) ────
bc_semibatch_cv = build_bc_config(n_comp=nc, P_out_bar=P_OUT, Cv=0.5)

# ── CSTR / paso continuo: v_out=None → calculado por continuidad molar ────────
bc_cstr = build_bc_config(
    n_comp=nc, P_out_bar=P_OUT,
    v_gas_in=V_IN, T_gas_in=T_IN, y_gas_in=y_air,
)

# ── updraft: gas z=0↑, sólido z=L↓ (contracorriente) ────────────────────────
bc_updraft = build_bc_config(
    n_comp=nc, P_out_bar=P_OUT,
    v_gas_in=V_IN, T_gas_in=T_IN, y_gas_in=y_air,
    v_solid=V_SOL, direction="updraft",
    rho_solid_in=rho_sol_in, T_solid_in=T_SOL,
)

# ── downdraft: gas y sólido cocorriente, ambos z=0↓ ──────────────────────────
bc_downdraft = build_bc_config(
    n_comp=nc, P_out_bar=P_OUT,
    v_gas_in=V_IN, T_gas_in=T_IN, y_gas_in=y_air,
    v_solid=V_SOL, direction="downdraft",
    rho_solid_in=rho_sol_in, T_solid_in=T_SOL,
)

# ── conveyor: caudal sólido calculado desde balance de masa ──────────────────
bc_conveyor = build_bc_config(
    n_comp=nc, P_out_bar=P_OUT,
    v_gas_in=V_IN, T_gas_in=T_IN, y_gas_in=y_air,
    v_solid=V_SOL, direction="updraft",
    T_solid_in=T_SOL,
    inlet_mode="computed", inlet_method="explicit",
    rho_solid_fresh_total=rho_p*(1-epsi_r), mc_wb=0.165,
)

# ── Tabla resumen ─────────────────────────────────────────────────────────────
def _v_str(v):
    if v is None:     return "None"
    if callable(v):   return "callable"
    return f"{float(v):.4g}"

def _v_out_str(bc):
    Cv = bc.get("Cv")
    v  = bc.get("v_out")
    if Cv is not None:
        return f"Cv={Cv:.4g}"
    if v is None:
        return "None  → continuidad"
    elif float(v) == 0.0:
        return "0.0   → sellado"
    else:
        return f"{float(v):.4g}  → v_vent_max"

rows = []
configs = [
    ("batch",          bc_batch),
    ("semibatch",      bc_semibatch),
    ("semibatch_cv",   bc_semibatch_cv),
    ("cstr",           bc_cstr),
    ("updraft",        bc_updraft),
    ("downdraft",      bc_downdraft),
    ("conveyor",       bc_conveyor),
]
for name, bc in configs:
    rows.append({
        "config":          name,
        "v_gas_in [m/s]":  _v_str(bc["v_gas_in"]),
        "outlet":          _v_out_str(bc),
        "v_solid [m/s]":   f"{bc['v_solid']:.4g}",
        "direction":       bc["direction"] if bc["direction"] else "-",
        "rho_solid_in":    ("array(3)" if bc["rho_solid_in"] is not None
                            else ("None — computed" if bc["inlet_mode"]=="computed"
                                  else "None")),
        "inlet_mode":      bc["inlet_mode"],
    })
df1 = pd.DataFrame(rows).set_index("config")
display(df1)
print("TEST 1 OK — 7 configuraciones construidas correctamente con build_bc_config")
print("  (batch, semibatch simple, semibatch ISA, CSTR, updraft, downdraft, conveyor)")

## TEST 2 — Condiciones térmicas de pared

El comportamiento de la pared depende de si el modelo de pared dinámica está activo.

### Geometría de la pared

| Símbolo | Significado |
|---|---|
| Ti (Tw_i) | Temperatura de la **superficie interior** de la pared (contacto con el gas) |
| To | Temperatura de la **superficie exterior** de la pared (contacto con el entorno) |

### 2a — `shell_tube = False` (sin `wall_config`)

La pared no tiene masa térmica propia. **Ti = To** en todo momento.
El vector de estado es `(16·N,)`. Las 4 condiciones de contorno actúan sobre la superficie interior:

| Modo | Qué prescribe sobre Ti | Parámetros clave |
|---|---|---|
| `adiabatic` | Sin intercambio — Ti libre | — |
| `heatfluxwall` | Flujo de calor desde el exterior hacia Ti | `Qwall [W]` |
| `fixed_twall` | Ti = T_wall = cte | `T_wall [K]` |
| `ambient_htc` | Convección desde To=Ti hacia el ambiente | `h_ambi`, `T_ambi`, `k_wall` |

### 2b — `shell_tube = True` (con `wall_config`)

La pared tiene masa térmica propia. **Tw_i (Ti) es una variable del ODE** integrada
junto con el gas y el sólido. El vector de estado crece a `(17·N,)`.
Las condiciones de contorno se aplican ahora sobre la superficie **exterior** (To):

| Modo | Qué prescribe sobre To | Compatible | Parámetros clave |
|---|---|---|---|
| `adiabatic` | Sin Q desde el exterior — To libre | ✅ | — |
| `heatfluxwall` | Flujo de calor desde exterior hacia To | ✅ | `Qwall [W]` |
| `fixed_twall` | To = T_wall = cte; Tw_i sigue siendo dinámica | ✅ | `T_wall [K]` |
| `ambient_htc` | Convección exterior sobre To | ✅ | `h_ambi`, `T_ambi`, `k_wall` |

> Todos los modos son compatibles con `shell_tube = True`. En `fixed_twall`,
> `T_wall` prescribe **To** (exterior), no Tw_i, por lo que no existe conflicto
> con la variable dinámica de la pared interior.

In [9]:
from src.units.gasifier.config.wall_c      import build_wall_config

K_SS, RHO_SS, CP_SS = 16.5, 7950.0, 510.0   # propiedades térmicas del acero inoxidable 316L

# ─────────────────────────────────────────────────────────────────────────────
# 2a — shell_tube = False  (sin wall_config, sv = 17*N)
# ─────────────────────────────────────────────────────────────────────────────
tbc_adiabatic = build_thermal_bc_config(mode="adiabatic",    Di=Di, Do=Do, e_wall=e_wall)
tbc_heatflux  = build_thermal_bc_config(mode="heatfluxwall", Di=Di, Do=Do, e_wall=e_wall,
                    Qwall=500.0, k_wall=K_SS, rho_wall=RHO_SS, Cp_wall=CP_SS)
tbc_fixed_T   = build_thermal_bc_config(mode="fixed_twall",  Di=Di, Do=Do, e_wall=e_wall,
                    T_wall=1100.0, k_wall=K_SS, rho_wall=RHO_SS, Cp_wall=CP_SS)
tbc_ambient   = build_thermal_bc_config(mode="ambient_htc",  Di=Di, Do=Do, e_wall=e_wall,
                    h_ambi=15.0, T_ambi=293.15, k_wall=K_SS, rho_wall=RHO_SS, Cp_wall=CP_SS)

tbcs = [("adiabatic",tbc_adiabatic),("heatfluxwall",tbc_heatflux),
         ("fixed_twall",tbc_fixed_T),("ambient_htc",tbc_ambient)]

rows_2a = []
for name, tbc in tbcs:
    rows_2a.append({
        "modo":            name,
        "actua sobre":     "Ti (interior)",
        "Qwall [W]":       tbc["Qwall"],
        "T_wall [K]":      tbc["T_wall"],
        "h_ambi [W/m2/K]": tbc["h_ambi"],
        "T_ambi [K]":      tbc["T_ambi"],
        "sv0.shape":       f"({17*N},)",
    })
print("=== 2a  shell_tube=False — 4 modos, actúan sobre Ti ===")
display(pd.DataFrame(rows_2a).set_index("modo"))

# ─────────────────────────────────────────────────────────────────────────────
# 2b — shell_tube = True  (con wall_config, sv = 18*N)
# ─────────────────────────────────────────────────────────────────────────────
wall_cfg = build_wall_config(
    N=N, Di=Di, Do=Do, T_w_init=300.0,
    material_id="SS316L", material_mode="polynomial",
    db_path=SOLID_DB, epsilon_wall=0.85,
)

_pg   = build_gas_prop_config(fuel_config=fuel_config, mode="constant", db_path=GAS_DB)
_sc   = build_solid_prop_config(fuel_config)
_tref = float(np.min(np.asarray(_pg["Tref"])))
_y0   = np.zeros(nc); _y0[-1] = 1.0
_init = build_initial_c_config(
    P_init=1.01325, Tg_init=300.0, Ts_init=300.0, y_init=_y0,
    rho_biomass_init=rho_p*(1-epsi_r), rho_char_init=1e-6,
    rho_moisture_init=0.165/0.835*rho_p*(1-epsi_r),
    n_comp=nc, N=N, prop_gas=_pg, epsi_r=epsi_r, gas_T_ref=_tref,
)
_tc = build_transport_config(mode="constant", N=N, n_comp=nc)

def _params_shell(tbc):
    return {
        "n_comp":nc,"N":N,"dz":dz,"Ai":Ai,"Di":Di,"Pi":Pi,"Po":Po,
        "prop_gas":_pg,"MW":np.asarray(_pg["MW"]),"gas_T_ref":_tref,
        "bc_config":bc_batch,"trans_config":_tc,
        "thermal_bc_config":tbc,"energy":True,
        "epsi_r":epsi_r,"dp0":dp0,"rho_char0":rho_char0,
        "fuel_config":fuel_config,"solid_config":_sc,"species":species,
        "wall_config":wall_cfg,"_cache":{},
    }

rows_2b = []
for name, tbc in tbcs:
    try:
        _validate_gasifier_params(_params_shell(tbc))
        result = "OK"
    except Exception as e:
        result = f"ERROR: {e}"
    note = ("To=T_wall fija; Tw_i dinámica" if name=="fixed_twall"
            else "To libre" if name=="adiabatic" else "")
    rows_2b.append({
        "modo":        name,
        "actua sobre": "To (exterior)",
        "compatible":  result,
        "sv0.shape":   f"({18*N},)" if result=="OK" else "—",
        "nota":        note,
    })

print("\n=== 2b  shell_tube=True — 4 modos, actúan sobre To; Tw_i = ODE ===")
print(f"wall_config: SS316L  A_w={wall_cfg['A_w']:.5f} m²  "
      f"T_w_init=300 K  epsilon={wall_cfg['epsilon_wall']}")
display(pd.DataFrame(rows_2b).set_index("modo"))
print("TEST 2 OK")

=== 2a  shell_tube=False — 4 modos, actúan sobre Ti ===


,actua sobre,Qwall [W],T_wall [K],h_ambi [W/m2/K],T_ambi [K],sv0.shape
modo,,,,,,
adiabatic,Ti (interior),NaN,NaN,NaN,NaN,"(80,)"
heatfluxwall,Ti (interior),500,NaN,NaN,NaN,"(80,)"
fixed_twall,Ti (interior),NaN,1100,NaN,NaN,"(80,)"
ambient_htc,Ti (interior),NaN,NaN,15,293.1,"(80,)"



=== 2b  shell_tube=True — 4 modos, actúan sobre To; Tw_i = ODE ===
wall_config: SS316L  A_w=0.00235 m²  T_w_init=300 K  epsilon=0.85


,actua sobre,compatible,sv0.shape,nota
modo,,,,
adiabatic,To (exterior),OK,"(85,)",To libre
heatfluxwall,To (exterior),OK,"(85,)",
fixed_twall,To (exterior),OK,"(85,)",To=T_wall fija; Tw_i dinámica
ambient_htc,To (exterior),OK,"(85,)",


TEST 2 OK


## TEST 3 — Coeficientes de transporte (`build_transport_config`)

| Modo | h_bed | h_wall | D_disp | Cuándo usar |
|---|---|---|---|---|
| `constant` | Valor fijo [W/m²/K] | Valor fijo [W/m²/K] | Fijo o 0 | Exploración rápida, sensibilidad |
| `correlation` | Ranz-Marshall: Nu = 2 + 1.1·Re⁰·⁶·Pr^(1/3) | Dittus-Boelter / Nu=3.66 | Bodenstein: D_ax = u·dp/Pe | Producción |

En modo `correlation` los coeficientes se recalculan en cada llamada al RHS
a partir de la temperatura y la velocidad local → capturan la variación axial.


In [10]:
# ── Construcción de las dos configuraciones ───────────────────────────────────
tc_const = build_transport_config(
    mode="constant", N=N, n_comp=nc,
    h_bed=80.0,    # [W/m²/K] coeficiente gas-partícula fijo
    h_wall=12.0,   # [W/m²/K] coeficiente gas-pared fijo
)

tc_corr = build_transport_config(
    mode="correlation", N=N, n_comp=nc,
    Pe_particle=2.0,    # número de Péclet de partícula (Bodenstein: D_ax = u*dp/Pe)
    Pe_plugflow=100.0,  # umbral Pe_bed por encima del cual se activa régimen plug-flow
)

# ── Ejemplo de coeficientes de correlación a un perfil de temperatura ─────────
# En modo 'correlation' h_bed y h_wall se recalculan en cada llamada al RHS
# a partir de la temperatura y velocidad local (Ranz-Marshall + Dittus-Boelter).
# Para ilustrar el rango de valores, se evalúan aquí las correlaciones para
# una corriente de N₂ puro a lo largo de un perfil de temperatura típico.

T_nodes = np.linspace(500.0, 1100.0, N)  # [K] perfil representativo (entrada fría → salida caliente)
v_ref   = float(V_IN)                    # [m/s] velocidad superficial de referencia
P_Pa    = P_OUT * 1e5                    # [Pa]  presión de referencia
R_GAS   = 8.31446                        # [J/mol/K]

# Propiedades de N₂ del modo polynomial (disponibles desde imports)
_pg_ref  = build_gas_prop_config(fuel_config=fuel_config, mode="polynomial", db_path=GAS_DB)
_N2      = species.index("N2")
MW_N2    = float(_pg_ref["MW"][_N2])            # [kg/mol]
_mu_fn   = _pg_ref["mu"][_N2]                   # callable µ(T) [Pa·s]
_k_fn    = _pg_ref["k"][_N2]                    # callable k(T) [W/m/K]
_Cp_fn   = _pg_ref["Cp_molar"][_N2]             # callable Cp(T) [J/mol/K]

mu_N2  = np.array([float(_mu_fn(T)) for T in T_nodes])    # [Pa·s]
k_N2   = np.array([float(_k_fn(T))  for T in T_nodes])    # [W/m/K]
Cp_N2  = np.array([float(_Cp_fn(T)) for T in T_nodes])    # [J/mol/K]

# Densidad del gas ideal + velocidad intersticial
rho_N2  = P_Pa * MW_N2 / (R_GAS * T_nodes)   # [kg/m³]
v_int   = v_ref / epsi_r                       # [m/s] velocidad intersticial

# Números adimensionales (Ranz-Marshall para partícula, Dittus-Boelter para tubo)
Pr     = (Cp_N2 / MW_N2) * mu_N2 / k_N2               # Pr = Cp[J/kg/K] · µ / k
Re_p   = rho_N2 * v_int * dp0 / mu_N2                 # Re partícula (escala dp)
Re_D   = rho_N2 * v_ref * Di  / mu_N2                 # Re tubería (escala Di)

Nu_bed  = 2 + 1.1 * np.maximum(Re_p, 0)**0.6 * np.maximum(Pr, 0)**(1/3)
Nu_wall = np.where(Re_D < 2300, 3.66, 0.023 * Re_D**0.8 * Pr**0.4)

h_bed_corr  = Nu_bed  * k_N2 / dp0   # [W/m²/K]
h_wall_corr = Nu_wall * k_N2 / Di    # [W/m²/K]

# ── Tabla 1: modo constant ────────────────────────────────────────────────────
print("=== Modo constant — valores fijos en todos los nodos ===")
df_const = pd.DataFrame({
    "Nodo":               np.arange(N),
    "T_ref [K]":          T_nodes.round(0),
    "h_bed [W/m²/K]":    np.full(N, 80.0),
    "h_wall [W/m²/K]":   np.full(N, 12.0),
}).set_index("Nodo")
display(df_const)

# ── Tabla 2: modo correlation ─────────────────────────────────────────────────
print(f"\n=== Modo correlation — Ranz-Marshall / Dittus-Boelter ===")
print(f"    (v_ref={v_ref} m/s, P={P_OUT:.5f} bar, N₂ puro como gas de referencia)")
df_corr = pd.DataFrame({
    "Nodo":               np.arange(N),
    "T [K]":              T_nodes.round(0),
    "Re_p [-]":           Re_p.round(1),
    "Pr [-]":             Pr.round(3),
    "Nu_bed [-]":         Nu_bed.round(2),
    "h_bed [W/m²/K]":    h_bed_corr.round(1),
    "Nu_wall [-]":        Nu_wall.round(2),
    "h_wall [W/m²/K]":   h_wall_corr.round(2),
}).set_index("Nodo")
display(df_corr)
print("TEST 3 OK — La correlación produce valores crecientes con T (mayor T → menor µ → mayor Re → mayor h_bed)")

=== Modo constant — valores fijos en todos los nodos ===


,T_ref [K],h_bed [W/m²/K],h_wall [W/m²/K]
Nodo,,,
0,500,80,12
1,650,80,12
2,800,80,12
3,950,80,12
4,1100,80,12



=== Modo correlation — Ranz-Marshall / Dittus-Boelter ===
    (v_ref=0.05 m/s, P=1.01325 bar, N₂ puro como gas de referencia)


,T [K],Re_p [-],Pr [-],Nu_bed [-],h_bed [W/m²/K],Nu_wall [-],h_wall [W/m²/K]
Nodo,,,,,,,
0,500,22.7,0.681,8.29,32.4,3.66,1.43
1,650,14.7,0.682,6.85,32.6,3.66,1.74
2,800,10.4,0.69,5.96,33.1,3.66,2.03
3,950,7.8,0.7,5.36,33.7,3.66,2.3
4,1100,6.2,0.71,4.92,34.5,3.66,2.56


TEST 3 OK — La correlación produce valores crecientes con T (mayor T → menor µ → mayor Re → mayor h_bed)


## TEST 4 — Propiedades del gas (`build_gas_prop_config`)

El gasificador tiene **9 especies fijas** en orden canónico:
`[CO, CO2, H2O, H2, O2, CH4, C2H4, tar, N2]`.
La especie `tar` no está en `gasdb.txt` — sus propiedades vienen del YAML del combustible
y se inyectan automáticamente en el wrapper del gasificador.

| Modo | µ, k, Cp, h | Cuándo usar |
|---|---|---|
| `constant` | Escalares evaluados a `T_ref` | Exploración rápida, sensibilidad |
| `polynomial` | Callables f(T) | Producción — captura variación con temperatura |

In [11]:
pg_const = build_gas_prop_config(fuel_config=fuel_config, mode="constant",   db_path=GAS_DB)
pg_poly  = build_gas_prop_config(fuel_config=fuel_config, mode="polynomial", db_path=GAS_DB)

# Tabla de propiedades invariantes con T (comunes a ambos modos)
df4 = pd.DataFrame({
    "Especie":      pg_const["species"],
    "MW [kg/mol]":  pg_const["MW"],
    "sigmaLJ [A]":  pg_const["sigmaLJ"],
    "epskB [K]":    pg_const["epskB"],
    "Tref [K]":     pg_const["Tref"],
    "Tmax [K]":     pg_const["Tmax"],
}).set_index("Especie")
pd.set_option("display.float_format", "{:.5g}".format)
print("=== Propiedades invariantes con T ===")
display(df4)

# Verificar callables en modo polynomial a T = 800 K
T_chk = 800.0
print(f"\n=== Propiedades en modo polynomial @ T = {T_chk:.0f} K ===")
df4b = pd.DataFrame({
    "Especie":         pg_poly["species"],
    "mu [µPa·s]":  [float(f(T_chk))*1e6 if callable(f) else float(f)*1e6
                   for f in pg_poly["mu"]],
    "k [mW/m·K]":  [float(f(T_chk))*1e3 if callable(f) else float(f)*1e3
                   for f in pg_poly["k"]],
    "Cp [J/mol·K]":[float(f(T_chk)) if callable(f) else float(f)
                   for f in pg_poly["Cp_molar"]],
    "h [kJ/mol]":  [float(f(T_chk))/1e3 if callable(f) else float(f)/1e3
                   for f in pg_poly["h_molar"]],
}).set_index("Especie")
display(df4b)
print("TEST 4 OK — 2 modos de gas construidos (constant + polynomial); tar incluido desde YAML")

=== Propiedades invariantes con T ===


,MW [kg/mol],sigmaLJ [A],epskB [K],Tref [K],Tmax [K]
Especie,,,,,
CO,0.028011,3.758,148.6,298,5000
CO2,0.04401,3.941,195.2,298,5000
H2O,0.018015,2.641,809.1,383,5000
H2,0.0020159,2.8227,59.7,298,5000
O2,0.031999,3.467,106.7,298,5000
CH4,0.016043,3.822,137,298,5000
C2H4,0.028054,4.163,224.7,298,5000
tar,0.0566,5.5,450,298,1000
N2,0.028014,3.798,71.4,298,5000



=== Propiedades en modo polynomial @ T = 800 K ===


,mu [µPa·s],k [mW/m·K],Cp [J/mol·K],h [kJ/mol]
Especie,,,,
CO,31.011,48.181,31.845,27.592
CO2,33.458,56.437,51.494,45.072
H2O,28.682,69.909,38.698,63.694
H2,17.041,383.87,29.666,22.608
O2,41.204,60.019,33.664,24.53
CH4,23.026,134.53,62.714,39.368
C2H4,23.181,78.016,83.835,43.416
tar,13.088,27.267,136.7,61.412
N2,34.163,55.514,31.399,23.731


TEST 4 OK — 2 modos de gas construidos (constant + polynomial); tar incluido desde YAML


## TEST 5 — Construcción y validación del dict `params` completo

Para cada modo operativo se construye el dict `params` completo que necesita
`run_step` y se valida con `_validate_gasifier_params` sin ejecutar ninguna
integración temporal.

La tabla muestra el tamaño del vector de estado `sv0` y los valores iniciales
clave para confirmar que la inicialización es coherente.

> `sv0.shape = (16·N,)` siempre que no se use `wall_config`.  
> Con `wall_config` activo, `sv0.shape = (17·N,)`.
>
> Layout: `[C(9×N) | ρ_s(3×N) | Hg(N) | Ts(N) | Q_mt_acc(N) | Q_rxn_acc(N)]` = 16·N DOFs

In [12]:
# Bloque común: gas props + solid props (no dependen del modo operativo)
prop_gas     = build_gas_prop_config(fuel_config=fuel_config, mode="polynomial", db_path=GAS_DB)
solid_config = build_solid_prop_config(fuel_config)
gas_T_ref    = float(np.min(np.asarray(prop_gas["Tref"])))   # [K] temperatura de referencia de entalpía
MW_arr       = np.asarray(prop_gas["MW"])                     # [kg/mol]

# Condiciones iniciales comunes a todos los casos
P_INIT          = 1.01325                             # [bar]      presión inicial
TG_INIT = TS_INIT = 300.0                             # [K]        temperatura inicial gas y sólido
rho_bio  = rho_p * (1 - epsi_r)                      # [kg/m³_bed] biomasa inicial en lecho
rho_moi  = 0.165 / (1 - 0.165) * rho_bio             # [kg/m³_bed] humedad inicial (16.5 % wb)
y0       = np.zeros(nc); y0[species.index("N2")] = 1.0  # composición inicial: 100 % N₂

# ── Índices en el vector de estado sv0 ───────────────────────────────────────
# Layout: [C(nc×N) | rho_bio(N) | rho_char(N) | rho_moi(N) | Hg(N) | Ts(N) | Q_mt_acc(N) | Q_rxn_acc(N)]
# Total:  (nc + 3 + 1 + 1 + 3) × N  =  17 × N  DOFs
IDX_C_N2    = (nc - 1) * N      # C_N2 celda 0    → índice (nc-1)×N = 8×N
IDX_rho_bio = nc * N            # rho_biomasa c.0 → índice nc×N
IDX_Hg      = (nc + 3) * N     # Hg celda 0      → índice (nc+3)×N
IDX_Ts      = (nc + 4) * N     # Ts celda 0      → índice (nc+4)×N

def base_params(bc_cfg, tbc_cfg, tc_cfg):
    init = build_initial_c_config(
        P_init=P_INIT, Tg_init=TG_INIT, Ts_init=TS_INIT, y_init=y0,
        rho_biomass_init=rho_bio, rho_char_init=1e-6, rho_moisture_init=rho_moi,
        n_comp=nc, N=N, prop_gas=prop_gas, epsi_r=epsi_r, gas_T_ref=gas_T_ref,
    )
    return {
        "n_comp":nc,"N":N,"dz":dz,"Ai":Ai,"Di":Di,"Pi":Pi,"Po":Po,
        "prop_gas":prop_gas,"MW":MW_arr,"gas_T_ref":gas_T_ref,
        "bc_config":bc_cfg,"trans_config":tc_cfg,"thermal_bc_config":tbc_cfg,
        "energy":True,
        "epsi_r":epsi_r,"dp0":dp0,"rho_char0":rho_char0,
        "fuel_config":fuel_config,"solid_config":solid_config,"species":species,
        "_cache":{},
    }, init["sv0"]

# Condición térmica y transporte comunes para todos los casos
tbc_ref = build_thermal_bc_config(
    mode="heatfluxwall", Di=Di, Do=Do, e_wall=e_wall,
    Qwall=500.0, k_wall=16.5, rho_wall=7950.0, Cp_wall=510.0)
tc_ref  = build_transport_config(mode="correlation", N=N, n_comp=nc)

rows5 = []
cases = [
    ("batch",    bc_batch),
    ("cstr",     bc_cstr),
    ("updraft",  bc_updraft),
    ("conveyor", bc_conveyor),
]
for name, bc_cfg in cases:
    p, sv0 = base_params(bc_cfg, tbc_ref, tc_ref)
    try:
        _validate_gasifier_params(p)
        status = "OK"
    except Exception as exc:
        status = f"ERROR: {exc}"
    rows5.append({
        "modo":                  name,
        "sv0.shape":             str(sv0.shape),
        "C_N2(c0) [mol/m³]":    round(float(sv0[IDX_C_N2]),    2),
        "rho_bio(c0) [kg/m³]":  round(float(sv0[IDX_rho_bio]), 1),
        "Hg(c0) [J/m³]":        round(float(sv0[IDX_Hg]),      1),
        "Ts(c0) [K]":           round(float(sv0[IDX_Ts]),       2),
        "validacion":            status,
    })

df5 = pd.DataFrame(rows5).set_index("modo")
pd.set_option("display.float_format", "{:.4g}".format)
display(df5)

n_ok = sum(1 for r in rows5 if r["validacion"]=="OK")
print(f"\nTEST 5 OK — {n_ok}/{len(rows5)} configuraciones validadas sin errores")
print(f"sv0.shape = {rows5[0]['sv0.shape']}  (17×N = {17*N})")
print(f"Layout sv0: C[{nc}sp×{N}celdas] | rho_s[3×{N}] | Hg[{N}] | Ts[{N}] | Q_acc[3×{N}]")

,sv0.shape,C_N2(c0) [mol/m³],rho_bio(c0) [kg/m³],Hg(c0) [J/m³],Ts(c0) [K],validacion
modo,,,,,,
batch,"(80,)",40.62,172,2.124e+05,300,OK
cstr,"(80,)",40.62,172,2.124e+05,300,OK
updraft,"(80,)",40.62,172,2.124e+05,300,OK
conveyor,"(80,)",40.62,172,2.124e+05,300,OK



TEST 5 OK — 4/4 configuraciones validadas sin errores
sv0.shape = (80,)  (16×N = 80)
Layout sv0: C[9sp×5celdas] | rho_s[3×5] | Hg[5] | Ts[5] | Q_acc[2×5]
